In [3]:
from pathlib import Path
import pandas as pd

project_root = Path(
    r"C:\Users\User\OneDrive\Desktop\malaysia-food-prices"
)
raw_dir = project_root / "data" / "raw"
staging_dir = project_root / "data" / "staging"
docs_dir = project_root / "docs"

staging_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)

basket_codes = [
    1, 1111, 904, 918, 1589,
    917, 129, 1564, 1131, 94,
    114, 103, 1431, 1555, 193,
]

months = pd.period_range("2022-12", "2025-12", freq="M")
start_date = pd.Timestamp("2022-12-25")
end_date = pd.Timestamp("2025-12-31")

# Check all inputs before writing anything.
missing = [
    f"pricecatcher_{month}.parquet"
    for month in months
    if not (raw_dir / f"pricecatcher_{month}.parquet").is_file()
]
assert not missing, f"Missing files: {missing}"

extraction_log = []

for month in months:
    source = raw_dir / f"pricecatcher_{month}.parquet"

    monthly = pd.read_parquet(
        source,
        columns=["date", "premise_code", "item_code", "price"],
    )
    source_rows = len(monthly)

    selected = monthly.loc[
        monthly["item_code"].isin(basket_codes)
    ].copy()
    del monthly

    selected["date"] = pd.to_datetime(
    selected["date"], errors="raise"
    ).astype("datetime64[ms]")

    # Stop rather than silently exclude records with missing dates.
    assert selected["date"].notna().all(), (
        f"Missing dates among selected items in {month}"
    )

    selected = selected.loc[
        selected["date"].between(start_date, end_date)
    ].copy()

    selected["source_month"] = str(month)
    selected["is_supporting_period"] = (
        selected["date"] < pd.Timestamp("2023-01-01")
    )

    destination = staging_dir / f"basket_{month}.parquet"
    selected.to_parquet(destination, index=False)

    # Verify the saved file against the extracted data.
    saved = pd.read_parquet(destination)
    pd.testing.assert_frame_equal(
        selected.reset_index(drop=True),
        saved.reset_index(drop=True),
    )

    extraction_log.append({
        "month": str(month),
        "source_rows": source_rows,
        "selected_rows": len(saved),
        "observed_items": saved["item_code"].nunique(),
        "observed_days": saved["date"].nunique(),
        "supporting_rows": int(saved["is_supporting_period"].sum()),
    })

    print(f"{month}: saved and verified {len(saved):,} rows")
    del selected, saved

extraction_log = pd.DataFrame(extraction_log)
extraction_log.to_csv(
    docs_dir / "basket_extraction_log.csv",
    index=False,
)

print("\nMonthly files saved:", len(extraction_log))
print("Total extracted records:",
      f"{extraction_log['selected_rows'].sum():,}")
print("Supporting records:",
      f"{extraction_log['supporting_rows'].sum():,}")
print("Saved-file verification: all passed")

2022-12: saved and verified 30,278 rows
2023-01: saved and verified 172,201 rows
2023-02: saved and verified 159,140 rows
2023-03: saved and verified 186,489 rows
2023-04: saved and verified 161,516 rows
2023-05: saved and verified 151,383 rows
2023-06: saved and verified 172,747 rows
2023-07: saved and verified 188,093 rows
2023-08: saved and verified 177,237 rows
2023-09: saved and verified 166,250 rows
2023-10: saved and verified 188,758 rows
2023-11: saved and verified 167,063 rows
2023-12: saved and verified 151,333 rows
2024-01: saved and verified 187,846 rows
2024-02: saved and verified 161,628 rows
2024-03: saved and verified 168,238 rows
2024-04: saved and verified 189,804 rows
2024-05: saved and verified 201,504 rows
2024-06: saved and verified 179,903 rows
2024-07: saved and verified 197,861 rows
2024-08: saved and verified 169,950 rows
2024-09: saved and verified 205,110 rows
2024-10: saved and verified 207,645 rows
2024-11: saved and verified 192,038 rows
2024-12: saved an

In [4]:
import numpy as np

quality_rows = []
key_columns = ["date", "premise_code", "item_code"]
original_columns = key_columns + ["price"]

for month in months:
    df = pd.read_parquet(
        staging_dir / f"basket_{month}.parquet"
    )

    quality_rows.append({
        "month": str(month),
        "rows": len(df),
        "missing_dates": int(df["date"].isna().sum()),
        "missing_item_codes": int(df["item_code"].isna().sum()),
        "missing_premise_codes": int(df["premise_code"].isna().sum()),
        "missing_prices": int(df["price"].isna().sum()),
        "infinite_prices": int(
            df["price"].isin([np.inf, -np.inf]).sum()
        ),
        "zero_prices": int(df["price"].eq(0).sum()),
        "negative_prices": int(df["price"].lt(0).sum()),
        "exact_duplicates": int(
            df.duplicated(subset=original_columns).sum()
        ),
        "repeated_keys": int(
            df.duplicated(subset=key_columns).sum()
        ),
        "dates_outside_source_month": int(
            df["date"].dt.to_period("M").ne(month).sum()
        ),
        "unexpected_item_codes": int(
            (~df["item_code"].isin(basket_codes)).sum()
        ),
    })

quality = pd.DataFrame(quality_rows)
quality.to_csv(
    docs_dir / "basket_quality_audit.csv",
    index=False,
)

issue_columns = quality.columns.drop(["month", "rows"])
problem_months = quality.loc[
    quality[issue_columns].gt(0).any(axis=1)
]

print("Months checked:", len(quality))
print("Rows checked:", f"{quality['rows'].sum():,}")

print("\nIssue totals:")
print(quality[issue_columns].sum().to_string())

print("\nMonths with issues:")
print(
    problem_months.to_string(index=False)
    if not problem_months.empty
    else "None"
)

Months checked: 37
Rows checked: 6,945,359

Issue totals:
missing_dates                  0
missing_item_codes             0
missing_premise_codes          0
missing_prices                 0
infinite_prices                0
zero_prices                    0
negative_prices                0
exact_duplicates              31
repeated_keys                 31
dates_outside_source_month     0
unexpected_item_codes          0

Months with issues:
  month  rows  missing_dates  missing_item_codes  missing_premise_codes  missing_prices  infinite_prices  zero_prices  negative_prices  exact_duplicates  repeated_keys  dates_outside_source_month  unexpected_item_codes
2022-12 30278              0                   0                      0               0                0            0                0                31             31                           0                      0


In [5]:
support = pd.read_parquet(
    staging_dir / "basket_2022-12.parquet"
)

key_columns = ["date", "premise_code", "item_code"]
original_columns = key_columns + ["price"]

# Save every occurrence in each duplicate group for inspection.
duplicate_group_rows = support.loc[
    support.duplicated(subset=original_columns, keep=False)
].sort_values(original_columns)

duplicate_group_rows.to_csv(
    docs_dir / "supporting_duplicate_records.csv",
    index=False,
)

# Preview deduplication in memory; do not overwrite any files.
deduplicated = support.drop_duplicates(
    subset=original_columns,
    keep="first",
)

print("Supporting rows before:", len(support))
print("Extra duplicate rows removed:", len(support) - len(deduplicated))
print("Supporting rows after:", len(deduplicated))
print(
    "Repeated keys remaining:",
    deduplicated.duplicated(subset=key_columns).sum(),
)
print("Saved duplicate audit: supporting_duplicate_records.csv")

Supporting rows before: 30278
Extra duplicate rows removed: 31
Supporting rows after: 30247
Repeated keys remaining: 0
Saved duplicate audit: supporting_duplicate_records.csv


In [6]:
dedup_dir = project_root / "data" / "interim" / "deduplicated"
dedup_dir.mkdir(parents=True, exist_ok=True)

dedup_log = []
original_columns = ["date", "premise_code", "item_code", "price"]
key_columns = ["date", "premise_code", "item_code"]

for month in months:
    df = pd.read_parquet(
        staging_dir / f"basket_{month}.parquet"
    )

    result = df.drop_duplicates(
        subset=original_columns,
        keep="first",
    ).reset_index(drop=True)

    assert not result.duplicated(subset=key_columns).any(), (
        f"Conflicting date–premise–item records in {month}"
    )

    destination = dedup_dir / f"basket_{month}.parquet"
    result.to_parquet(destination, index=False)

    saved = pd.read_parquet(destination)
    pd.testing.assert_frame_equal(result, saved)

    dedup_log.append({
        "month": str(month),
        "rows_before": len(df),
        "duplicates_removed": len(df) - len(result),
        "rows_after": len(result),
    })

dedup_log = pd.DataFrame(dedup_log)
dedup_log.to_csv(
    docs_dir / "basket_deduplication_log.csv",
    index=False,
)

print("Files saved:", len(dedup_log))
print("Duplicates removed:", dedup_log["duplicates_removed"].sum())
print("Remaining records:", f"{dedup_log['rows_after'].sum():,}")
print("Saved-file verification: all passed")

Files saved: 37
Duplicates removed: 31
Remaining records: 6,945,328
Saved-file verification: all passed


In [7]:
# Use the main analysis period only: January 2023–December 2025.
price_data = pd.concat(
    [
        pd.read_parquet(
            dedup_dir / f"basket_{month}.parquet",
            columns=["item_code", "price"],
        )
        for month in pd.period_range("2023-01", "2025-12", freq="M")
    ],
    ignore_index=True,
)

grouped = price_data.groupby("item_code")["price"]

summary = grouped.agg(
    observations="size",
    minimum="min",
    median="median",
    maximum="max",
)

percentiles = grouped.quantile([0.01, 0.99]).unstack()
percentiles.columns = ["p1", "p99"]

summary = summary.join(percentiles).reset_index()

items = pd.read_parquet(
    raw_dir / "lookup_item.parquet",
    columns=["item_code", "item", "unit"],
)

summary = summary.merge(
    items,
    on="item_code",
    how="left",
    validate="one_to_one",
)

summary = summary[
    [
        "item_code", "item", "unit", "observations",
        "minimum", "p1", "median", "p99", "maximum",
    ]
].sort_values("item_code")

summary.to_csv(
    docs_dir / "basket_price_summary_2023_2025.csv",
    index=False,
)

print(summary.to_string(index=False))
print("\nItems:", len(summary))
print("Main-period records:", f"{summary['observations'].sum():,}")

del price_data

 item_code                                                                               item    unit  observations  minimum    p1  median   p99  maximum
         1                                                             AYAM BERSIH - STANDARD     1kg        727819     0.86  6.70    8.99 12.00   880.00
        94                                                                CILI MERAH - MINYAK     1kg        556184     1.49  6.99   14.90 27.90    71.50
       103                                                        SANTAN KELAPA SEGAR (BIASA)     1kg        201281     0.55  4.30    8.69 18.00    80.00
       114                                                                             TOMATO     1kg        916625     0.03  1.90    5.99 12.00   399.00
       129                                                        BAWANG BESAR KUNING/HOLLAND     1kg        956751     0.15  1.98    4.80  8.00   660.00
       193                                                   SARDIN CAP KING

In [8]:
# Keep small monthly samples, then find extremes across all months.
samples = []

for month in pd.period_range("2023-01", "2025-12", freq="M"):
    df = pd.read_parquet(
        dedup_dir / f"basket_{month}.parquet",
        columns=["date", "premise_code", "item_code", "price"],
    )

    for code, group in df.groupby("item_code"):
        for label, sample in [
            ("lowest", group.nsmallest(3, "price")),
            ("highest", group.nlargest(3, "price")),
        ]:
            sample = sample.copy()
            sample["extreme"] = label
            samples.append(sample)

    del df

candidates = pd.concat(samples, ignore_index=True)
review_parts = []

for (code, label), group in candidates.groupby(
    ["item_code", "extreme"]
):
    sample = (
        group.nsmallest(3, "price")
        if label == "lowest"
        else group.nlargest(3, "price")
    )
    review_parts.append(sample)

extreme_review = pd.concat(review_parts, ignore_index=True)

extreme_review = extreme_review.merge(
    items[["item_code", "item", "unit"]],
    on="item_code",
    how="left",
    validate="many_to_one",
)

extreme_review.to_csv(
    docs_dir / "basket_extreme_price_review.csv",
    index=False,
)

print("Review rows:", len(extreme_review))
print("Items:", extreme_review["item_code"].nunique())

# Start with four examples to keep the output manageable.
print(
    extreme_review.loc[
        extreme_review["item_code"].isin([1, 917, 1111, 1555]),
        ["item_code", "extreme", "date", "premise_code", "price"],
    ].to_string(index=False)
)

Review rows: 90
Items: 15
 item_code extreme       date  premise_code   price
         1 highest 2025-06-26           424  880.00
         1 highest 2025-10-05           766   80.02
         1 highest 2025-12-09          6136   17.00
         1  lowest 2025-10-22         21823    0.86
         1  lowest 2025-12-28         21036    0.89
         1  lowest 2025-01-02         21207    2.22
       917 highest 2025-01-20         11621  318.00
       917 highest 2025-02-10          9548   37.70
       917 highest 2025-07-07         21108   30.90
       917  lowest 2024-07-01         18481    0.37
       917  lowest 2023-09-18         18017    0.39
       917  lowest 2025-06-05         17980    1.79
      1111 highest 2025-06-11           424 1140.00
      1111 highest 2025-02-17          5093 1110.00
      1111 highest 2025-09-08         18107   19.50
      1111  lowest 2025-11-01         11925    1.00
      1111  lowest 2024-09-02          5377    1.10
      1111  lowest 2025-08-21         

In [9]:
# Investigate four examples first.
cases = pd.DataFrame([
    (1, 424, "2025-06-26"),
    (917, 11621, "2025-01-20"),
    (1111, 424, "2025-06-11"),
    (1555, 17774, "2023-08-19"),
], columns=["item_code", "premise_code", "flagged_date"])

cases["flagged_date"] = pd.to_datetime(cases["flagged_date"])
context_parts = []

for case in cases.itertuples(index=False):
    start = case.flagged_date - pd.Timedelta(days=14)
    end = case.flagged_date + pd.Timedelta(days=14)
    parts = []

    for month in pd.period_range(start, end, freq="M"):
        df = pd.read_parquet(
            dedup_dir / f"basket_{month}.parquet",
            columns=["date", "premise_code", "item_code", "price"],
        )

        mask = (
            df["item_code"].eq(case.item_code)
            & df["premise_code"].eq(case.premise_code)
            & df["date"].between(start, end)
        )
        parts.append(df.loc[mask].copy())

    context = pd.concat(parts, ignore_index=True).sort_values("date")
    context["flagged_date"] = case.flagged_date
    context["is_flagged_date"] = context["date"].eq(case.flagged_date)

    assert context["is_flagged_date"].sum() == 1
    context_parts.append(context)

    print(
        f"\nItem {case.item_code}, premise {case.premise_code}"
        f" — reviewing {case.flagged_date.date()}"
    )
    print(
        context[["date", "price", "is_flagged_date"]]
        .to_string(index=False)
    )

price_context = pd.concat(context_parts, ignore_index=True)
price_context.to_csv(
    docs_dir / "extreme_price_context_examples.csv",
    index=False,
)

print("\nCases checked:", len(cases))
print("Flagged records found:", price_context["is_flagged_date"].sum())


Item 1, premise 424 — reviewing 2025-06-26
      date  price  is_flagged_date
2025-06-12   10.0            False
2025-06-13   10.0            False
2025-06-14   10.0            False
2025-06-15   10.0            False
2025-06-17    9.0            False
2025-06-18    8.7            False
2025-06-19    8.7            False
2025-06-20    8.7            False
2025-06-21    8.8            False
2025-06-22    8.8            False
2025-06-24    8.8            False
2025-06-25    8.8            False
2025-06-26  880.0             True
2025-06-28    8.8            False
2025-06-29    8.8            False
2025-07-01    8.8            False
2025-07-02    9.0            False
2025-07-03    9.0            False
2025-07-04    8.8            False
2025-07-05    8.8            False
2025-07-06    8.8            False
2025-07-08    8.8            False
2025-07-09    8.8            False
2025-07-10    8.8            False

Item 917, premise 11621 — reviewing 2025-01-20
      date  price  is_flagged_dat

In [10]:
suspected_errors = pd.DataFrame([
    ("2025-06-26", 1, 424, 880.00),
    ("2025-01-20", 917, 11621, 318.00),
    ("2025-06-11", 1111, 424, 1140.00),
    ("2023-08-19", 1555, 17774, 3599.00),
    ("2023-08-20", 1555, 17774, 3599.00),
], columns=["date", "item_code", "premise_code", "recorded_price"])

suspected_errors["date"] = pd.to_datetime(suspected_errors["date"])
suspected_errors["review_status"] = "suspected_recording_error"
suspected_errors["reason"] = (
    "Extreme price relative to nearby observations "
    "for the same item and premise"
)
suspected_errors["evidence_file"] = "extreme_price_context_examples.csv"
suspected_errors["action"] = "flag_only_pending_cleaning_policy"

assert not suspected_errors.duplicated(
    ["date", "item_code", "premise_code"]
).any()

suspected_errors.to_csv(
    docs_dir / "reviewed_price_anomalies.csv",
    index=False,
)

print("Suspected observations recorded:", len(suspected_errors))
print("Price values changed: 0")

Suspected observations recorded: 5
Price values changed: 0


In [11]:
flagged_dir = project_root / "data" / "interim" / "price_flagged"
flagged_dir.mkdir(parents=True, exist_ok=True)

screening_parts = []

for month in months:
    df = pd.read_parquet(
        dedup_dir / f"basket_{month}.parquet"
    )

    df["item_month_median"] = (
        df.groupby("item_code")["price"].transform("median")
    )
    df["price_to_median"] = (
        df["price"] / df["item_month_median"]
    )

    df["flag_low_price"] = df["price_to_median"].lt(0.2)
    df["flag_high_price"] = df["price_to_median"].gt(5)
    df["flag_extreme_price"] = (
        df["flag_low_price"] | df["flag_high_price"]
    )

    # Save all records, including flagged ones.
    df.to_parquet(
        flagged_dir / f"basket_{month}.parquet",
        index=False,
    )

    monthly_summary = (
        df.groupby("item_code")
        .agg(
            records=("price", "size"),
            low_flags=("flag_low_price", "sum"),
            high_flags=("flag_high_price", "sum"),
        )
        .reset_index()
    )
    monthly_summary["month"] = str(month)
    screening_parts.append(monthly_summary)

screening = pd.concat(screening_parts, ignore_index=True)
screening.to_csv(
    docs_dir / "price_screening_by_item_month.csv",
    index=False,
)

totals = (
    screening.groupby("item_code")
    [["records", "low_flags", "high_flags"]]
    .sum()
)

totals["total_flags"] = totals["low_flags"] + totals["high_flags"]
totals["flagged_percent"] = (
    100 * totals["total_flags"] / totals["records"]
).round(3)

print(totals.to_string())
print("\nMonths screened:", screening["month"].nunique())
print("Records screened:", f"{totals['records'].sum():,}")
print("Records flagged:", totals["total_flags"].sum())
print("Records removed: 0")

           records  low_flags  high_flags  total_flags  flagged_percent
item_code                                                              
1           731401          2           2            4            0.001
94          558562          9           1           10            0.002
103         202405          6           2            8            0.004
114         920665        123           3          126            0.014
129         961308        154           7          161            0.017
193         111683          0           2            2            0.002
904         148946          3           0            3            0.002
917         172968          2           3            5            0.003
918         122340          0          16           16            0.013
1111        361457         12           2           14            0.004
1131        822641        188           4          192            0.023
1431        100616          0           0            0          

In [12]:
flagged_parts = []

for month in months:
    df = pd.read_parquet(
        flagged_dir / f"basket_{month}.parquet"
    )

    flagged = df.loc[df["flag_extreme_price"]].copy()
    flagged["flag_direction"] = np.where(
        flagged["flag_low_price"], "low", "high"
    )
    flagged_parts.append(flagged)

flagged_records = pd.concat(flagged_parts, ignore_index=True)

# Retain the full review list locally.
flagged_records.to_csv(
    flagged_dir / "flagged_records_review.csv",
    index=False,
)

flag_groups = (
    flagged_records.groupby(
        ["item_code", "source_month", "flag_direction"]
    )
    .agg(
        flagged_records=("price", "size"),
        premises=("premise_code", "nunique"),
        recorded_days=("date", "nunique"),
        minimum_price=("price", "min"),
        maximum_price=("price", "max"),
        item_month_median=("item_month_median", "first"),
    )
    .reset_index()
    .sort_values("flagged_records", ascending=False)
)

flag_groups.to_csv(
    docs_dir / "price_flag_review_groups.csv",
    index=False,
)

print("Flagged records:", len(flagged_records))
print("Grouped records reconcile:",
      flag_groups["flagged_records"].sum() == len(flagged_records))

print("\nLargest 15 groups:")
print(flag_groups.head(15).to_string(index=False))

Flagged records: 587
Grouped records reconcile: True

Largest 15 groups:
 item_code source_month flag_direction  flagged_records  premises  recorded_days  minimum_price  maximum_price  item_month_median
      1131      2025-11            low              162        18             14           0.30           0.49               2.99
       129      2024-05            low              108        15              8           0.99           0.99               5.00
       129      2024-06            low               31        15              4           0.99           0.99               5.00
       114      2025-10            low               29         8              6           0.99           0.99               4.99
      1131      2025-10            low               15        15              1           0.49           0.49               2.99
       114      2024-12            low               14         7              5           0.99           1.00               5.69
       114      2

In [13]:
premises = pd.read_parquet(
    raw_dir / "lookup_premise.parquet"
)

focus = flagged_records.loc[
    flagged_records["flag_low_price"]
    & (
        (
            flagged_records["item_code"].eq(1131)
            & flagged_records["source_month"].eq("2025-11")
        )
        | (
            flagged_records["item_code"].eq(129)
            & flagged_records["source_month"].eq("2024-05")
        )
    )
].copy()

focus = focus.merge(
    premises[
        ["premise_code", "premise", "state", "premise_type"]
    ],
    on="premise_code",
    how="left",
    validate="many_to_one",
)

focus["premise_type"] = focus["premise_type"].str.strip()

shop_summary = (
    focus.groupby(
        ["item_code", "premise_code", "premise",
         "state", "premise_type"],
        dropna=False,
    )
    .agg(
        flagged_records=("price", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        minimum_price=("price", "min"),
        maximum_price=("price", "max"),
    )
    .reset_index()
    .sort_values(["item_code", "state", "premise_code"])
)

shop_summary.to_csv(
    docs_dir / "low_price_cluster_review.csv",
    index=False,
)

print(shop_summary.to_string(index=False))
print("\nRecords reviewed:", shop_summary["flagged_records"].sum())
print("Item–premise groups:", len(shop_summary))

 item_code  premise_code                                 premise           state             premise_type  flagged_records first_date  last_date  minimum_price  maximum_price
       129         17100 MYDIN WHOLESALE HYPERMARKET KULAI UTAMA           Johor              Hypermarket                8 2024-05-24 2024-05-31           0.99           0.99
       129         18063          MYDIN HYPERMARKET MUTIARA RINI           Johor              Hypermarket                8 2024-05-24 2024-05-31           0.99           0.99
       129         20786         MYDIN HYPERMARKET TAMAN RINTING           Johor              Hypermarket                8 2024-05-24 2024-05-31           0.99           0.99
       129         13715             MYDIN WHOLESALE HYPERMARKET        Kelantan              Hypermarket                8 2024-05-24 2024-05-31           0.99           0.99
       129         20731                      MYDIN MALL TUNJONG        Kelantan              Hypermarket                7 20

In [14]:
comparison_parts = []

for month in months:
    df = pd.read_parquet(
        flagged_dir / f"basket_{month}.parquet"
    )

    all_records = df.groupby("item_code").agg(
        records_all=("price", "size"),
        mean_all=("price", "mean"),
        median_all=("price", "median"),
        flagged_records=("flag_extreme_price", "sum"),
    )

    unflagged = (
        df.loc[~df["flag_extreme_price"]]
        .groupby("item_code")
        .agg(
            records_unflagged=("price", "size"),
            mean_unflagged=("price", "mean"),
            median_unflagged=("price", "median"),
        )
    )

    comparison = all_records.join(unflagged).reset_index()
    comparison["month"] = str(month)

    # Percentage change when flagged records are excluded.
    for measure in ["mean", "median"]:
        comparison[f"{measure}_change_pct"] = (
            (
                comparison[f"{measure}_unflagged"]
                / comparison[f"{measure}_all"]
            ) - 1
        ) * 100

    comparison_parts.append(comparison)

sensitivity = pd.concat(comparison_parts, ignore_index=True)

assert (
    sensitivity["records_all"]
    - sensitivity["records_unflagged"]
).eq(sensitivity["flagged_records"]).all()

sensitivity.to_csv(
    docs_dir / "price_flag_sensitivity.csv",
    index=False,
)

affected = sensitivity.loc[
    sensitivity["flagged_records"].gt(0)
].copy()

affected["absolute_mean_change"] = (
    affected["mean_change_pct"].abs()
)

columns = [
    "month", "item_code", "flagged_records",
    "mean_all", "mean_unflagged", "mean_change_pct",
    "median_all", "median_unflagged", "median_change_pct",
]

print("Largest 10 changes in monthly means:")
print(
    affected.sort_values("absolute_mean_change", ascending=False)
    [columns].head(10).round(3).to_string(index=False)
)

print("\nRecords accounted for:",
      f"{sensitivity['records_all'].sum():,}")
print("Flags accounted for:", sensitivity["flagged_records"].sum())
print("Count reconciliation: passed")

Largest 10 changes in monthly means:
  month  item_code  flagged_records  mean_all  mean_unflagged  mean_change_pct  median_all  median_unflagged  median_change_pct
2025-04        193                1     9.384           9.092           -3.119        9.00              9.00                0.0
2024-02        918                4     2.545           2.500           -1.780        2.50              2.50                0.0
2025-01        917                1     3.484           3.427           -1.611        3.40              3.40                0.0
2025-01       1589                1     2.894           2.850           -1.530        2.85              2.85                0.0
2023-08       1555                2    33.432          32.984           -1.340       32.90             32.90                0.0
2024-04        918                3     2.532           2.500           -1.281        2.50              2.50                0.0
2025-11        129                2     3.775           3.741      

In [15]:
median_changed = ~np.isclose(
    sensitivity["median_all"],
    sensitivity["median_unflagged"],
    rtol=0,
    atol=1e-10,
)

print("Item-month groups checked:", len(sensitivity))
print("Groups with changed medians:", median_changed.sum())
print(
    "Largest absolute median change (%):",
    round(sensitivity["median_change_pct"].abs().max(), 4),
)

print("\nGroups with changed medians:")
print(
    sensitivity.loc[
        median_changed,
        [
            "month", "item_code", "flagged_records",
            "median_all", "median_unflagged", "median_change_pct",
        ],
    ].to_string(index=False)
    if median_changed.any()
    else "None"
)

Item-month groups checked: 555
Groups with changed medians: 0
Largest absolute median change (%): 0.0

Groups with changed medians:
None


In [16]:
prepared_dir = project_root / "data" / "processed"
prepared_dir.mkdir(parents=True, exist_ok=True)

items = pd.read_parquet(raw_dir / "lookup_item.parquet")
premises = pd.read_parquet(raw_dir / "lookup_premise.parquet")

# Lookup keys must be present and unique to prevent duplicated price rows.
for lookup, key in [
    (items, "item_code"),
    (premises, "premise_code"),
]:
    assert lookup[key].notna().all(), f"Missing lookup key: {key}"
    assert lookup[key].is_unique, f"Repeated lookup key: {key}"

premises["premise_type_clean"] = premises["premise_type"].str.strip()

join_log = []

for month in months:
    df = pd.read_parquet(
        flagged_dir / f"basket_{month}.parquet"
    )
    rows_before = len(df)

    prepared = df.merge(
        items[["item_code", "item", "unit", "item_group", "item_category"]],
        on="item_code",
        how="left",
        validate="many_to_one",
        indicator="item_match",
    )

    prepared = prepared.merge(
        premises[
            ["premise_code", "premise", "state", "district",
             "premise_type", "premise_type_clean"]
        ],
        on="premise_code",
        how="left",
        validate="many_to_one",
        indicator="premise_match",
    )

    prepared["item_lookup_missing"] = prepared["item_match"].eq("left_only")
    prepared["premise_lookup_missing"] = (
        prepared["premise_match"].eq("left_only")
    )
    prepared = prepared.drop(columns=["item_match", "premise_match"])

    assert len(prepared) == rows_before, f"Row count changed: {month}"
    pd.testing.assert_frame_equal(
        df.reset_index(drop=True),
        prepared[df.columns].reset_index(drop=True),
    )

    destination = prepared_dir / f"basket_{month}.parquet"
    prepared.to_parquet(destination, index=False)

    saved = pd.read_parquet(destination)
    pd.testing.assert_frame_equal(prepared, saved)

    join_log.append({
        "month": str(month),
        "rows_before": rows_before,
        "rows_after": len(saved),
        "unmatched_items": int(saved["item_lookup_missing"].sum()),
        "unmatched_premises": int(saved["premise_lookup_missing"].sum()),
        "missing_states": int(saved["state"].isna().sum()),
        "missing_retailer_types": int(
            saved["premise_type_clean"].isna().sum()
        ),
        "flagged_prices": int(saved["flag_extreme_price"].sum()),
    })

    print(f"{month}: enriched and verified {len(saved):,} records")
    del df, prepared, saved

join_log = pd.DataFrame(join_log)
join_log.to_csv(
    docs_dir / "basket_lookup_join_audit.csv",
    index=False,
)

print("\nJoin totals:")
print(join_log.drop(columns="month").sum().to_string())
print("\nMonthly files saved:", len(join_log))
print("Original columns and saved files verified: all passed")

2022-12: enriched and verified 30,247 records
2023-01: enriched and verified 172,201 records
2023-02: enriched and verified 159,140 records
2023-03: enriched and verified 186,489 records
2023-04: enriched and verified 161,516 records
2023-05: enriched and verified 151,383 records
2023-06: enriched and verified 172,747 records
2023-07: enriched and verified 188,093 records
2023-08: enriched and verified 177,237 records
2023-09: enriched and verified 166,250 records
2023-10: enriched and verified 188,758 records
2023-11: enriched and verified 167,063 records
2023-12: enriched and verified 151,333 records
2024-01: enriched and verified 187,846 records
2024-02: enriched and verified 161,628 records
2024-03: enriched and verified 168,238 records
2024-04: enriched and verified 189,804 records
2024-05: enriched and verified 201,504 records
2024-06: enriched and verified 179,903 records
2024-07: enriched and verified 197,861 records
2024-08: enriched and verified 169,950 records
2024-09: enric